# IMPORTANT
**Run the next block of code for the notebook**
> If any error occurs with the block of code run the Requirements Section

# Detect Minino
The first step in the configuration process is to select the correct serial port to which the Minino device is connected.

<b>Identify the Port:</b> To check the available ports and determine the correct one, run the following code block:


In [1]:
import sys
sys.path.append("lib")

import ipywidgets as widgets
from IPython.display import display, clear_output

from main import Notebook, SerialConnection

notebook = Notebook()
serial_conn = SerialConnection()


scan_button = widgets.Button(description="Scan Ports")
detect_button = widgets.Button(description="Detect Minino", icon="arrow-right")
dropdown_ports = widgets.Dropdown(description='Ports:', layout=widgets.Layout(width='20%'))
detect_minino_output = widgets.Output(layout={'border': '1px solid black'})

def scan_ports(b):
    dropdown_ports.options = notebook.detect_serial_ports(0)

def detect_minino(btn):
    
    if not serial_conn.connect(port=dropdown_ports.value):
        with detect_minino_output:
            clear_output() 
            print("Error conecting the port")
        return
        
    data = serial_conn.send_command_string_with_response('h')
    
    if "minino" in data:
        with detect_minino_output:
            clear_output() 
            print(f"Minino Detected at {dropdown_ports.value}")
    else:
        with detect_minino_output:
            clear_output() 
            print("Minino no detected")
        
    serial_conn.disconnect()

scan_button.on_click(scan_ports)
detect_button.on_click(detect_minino)

display(
    widgets.Box([scan_button, dropdown_ports, detect_button]),
    detect_minino_output
)

Box(children=(Button(description='Scan Ports', style=ButtonStyle()), Dropdown(description='Ports:', layout=Lay…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

# Hands on Lab #4  Minino - Wifi Captive Portal
The Wifi Captive Portal enables the Minino device to function as a Wi-Fi Access Point (AP), allowing it to intercept client connections and present a fully customizable web interface. This brings enhanced flexibility, user data handling, and configuration options, making it easier to create engaging and effective captive experiences.

## Objetives

- <b>Explain the concept</b> of a rogue access point and its use in credential harvesting attacks.
- <b>Configure Minino</b> to host a captive portal that mimics a legitimate Wi‑Fi network.
- <b>Capture HTTP credentials</b> and probe requests from automatically connecting devices.
- <b>Analyze</b> the behavior of client devices during rogue AP interactions.
- <b>Log</b> and interpret the captured credential data for reporting or forensic purposes.
- <b>Evaluate</b> the security risks posed by untrusted Wi‑Fi networks and rogue APs.


## Walk-Through
#### What Is an Access Point?
A Wi-Fi access point (AP) is a networking device that allows wireless devices to connect to a wired network using Wi-Fi.

#### A Rogue Access Point
A rogue access point is an unauthorized wireless access point connected to a private network, creating a significant security risk. These can be set up by malicious actors or even by employees without permission, providing a backdoor for attackers to steal data, spread malware, or launch further attacks

### Naming the Captive Portal
The <b>SSID</b> (Service Set Identifier) is the human-readable name you assign to the Wi-Fi network (Access Point) hosted by Minino via the USB interface.
You could provide an SSID for the Access Point (AP). When the Captive Portal is executed, Minino will create a Wi-Fi AP using this assigned name, allowing client devices to connect to it.
Execute the following block of code, it will display a button and text box, write the name for the SSID and configure the name clicking on the button 

In [2]:
button_send_name = widgets.Button(description="send name")
text_input = widgets.Text()
respond_serial = widgets.Output(layout={'border': '1px solid black'})

def send_captive_name(b=None):

    with respond_serial:
        clear_output()  # Clear display

        name = text_input.value
        if not name:
            print("Please write a name")
            return

        if not serial_conn.connect(port=dropdown_ports.value):
            print("ERROR while connecting serial")
            return
        else:
            print(f"Device at {dropdown_ports.value} connected")

        # First send a help
        serial_conn.send_command_string("help")

        command_complete = f"captive_name {name}"
        response = serial_conn.send_command_string_with_response(command_complete)

        print("Response:", response)

        # Disconnect Serial
        if serial_conn.disconnect():
            print(f"Device at {dropdown_ports.value} disconnected")
        else:
            print("ERROR AT DISCONNECT")

button_send_name.on_click(send_captive_name)

display(widgets.HBox([text_input, button_send_name]), respond_serial)

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

### Running the Captive Portal
<b>Launch the Captive Portal Menu:</b> Minino does not feature a direct command to immediately generate the Wi-Fi AP. Instead, you must first execute the command that launches the configuration menu for the Captive Portal feature. On the next block of code, after the execution, it will send you to the Captive Portal configuration menu there navigate to the Run option using the buttons (up/down) and execute the option by pressing the right button (or the designated execution button).

In [4]:
import time
import ipywidgets as widgets
from IPython.display import display, clear_output

connect_button = widgets.Button(description='Connect', button_style="success")
captive_portal_menu = widgets.Button(description='Captive Portal')
display_error = widgets.Output(layout={'border': '1px solid black'})
is_connected = False

def button_function(btn):
    global is_connected

    if not is_connected:
        if not serial_conn.connect(port=dropdown_ports.value):
            with display_error:
                clear_output()
                print("Error connecting the port")
            return

        time.sleep(1)
        connect_button.description = 'Disconnect'
        connect_button.button_style = 'danger'
        is_connected = True

    else: 
        serial_conn.disconnect()
        time.sleep(1)
        connect_button.description = 'Connect'
        connect_button.button_style = 'success'
        is_connected = False


def go_to_captive_portal(btn):
    global is_connected
    if not is_connected:
        with display_error:
            clear_output()
            print("You must connect first.")
        return

    serial_conn.send_command_string("launch captive_portal")


connect_button.on_click(button_function)
captive_portal_menu.on_click(go_to_captive_portal)

display(
    widgets.Box([connect_button, dropdown_ports]),
    captive_portal_menu,
    display_error
)

Box(children=(Button(button_style='success', description='Connect', style=ButtonStyle()), Dropdown(description…

Button(description='Captive Portal', style=ButtonStyle())

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

On the following image is shown the example of the menu where we are going to launch:


### Verification: Checking the Active AP
Once the Captive Portal has been executed, you must verify that the Access Point (AP) is successfully broadcasting the signal.

You can check this using any Wi-Fi enabled device:
- Scan for Networks: Use your mobile phone or computer to scan the surrounding Wi-Fi networks.
<div style="text-align: center;">
  <img src="static/deauth_menu.jpeg" width="200" />
</div>

- Locate the SSID: You should see a new Access Point listed with the SSID name you previously configured for the Minino device.

<div style="text-align: center;">
  <img src="static/Captive_Portal_Scan_SSID.jpeg" width="200" />
</div>

If the network appears, the Captive Portal is active and ready to intercept client connections.

### Obteinning Data from the Captive Portal
Once you connect to the Minino Access Point, it will redirect you to a page, there you can introduce credentials.

<div style="text-align: center;">
  <img src="static/Captive_Portal_Cellphone.jpeg" width="200" />
</div>

After that, on your minino will be displayed the introduced credentials.

<div style="text-align: center;">
  <img src="static/Captive_Portal_Capture_credentials.jpeg" width="200" />
</div>

## Quick references
- **Captive Portal:** Unauthorized Wi‑Fi access point set up to trick users into connecting, often for credential capture or traffic interception.
- **Rogue Access Point:** Web page presented to users before granting internet access; in attacks, it can be used to harvest credentials.
- **Probe Request:** Wi‑Fi frame sent by a client to discover known networks; can reveal preferred SSIDs and past connections.
- **HTTP Credentials:** Usernames, passwords, or other authentication data sent in plaintext via HTTP; easily intercepted if unencrypted.
- **SSID:** Network name broadcast by a Wi‑Fi access point.